# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2 — Refresh / Content Opportunity Scoring** (provisional).

I want a ranked review queue that helps an editor decide which existing pages to look at first for refresh, expansion, CTR review, engagement review, or monitoring. The starter pipeline already frames this as “right page to review first, given limited capacity,” and the warehouse daily facts can later support a stronger future-window label. Ranking Signal Analysis would teach associations but stops short of a queue someone acts on; clustering describes types without a priority list; CTR-only scoring is narrower than the refresh mix I care about. Freestyle growth/recovery prediction is attractive later, but I want the decision and action clear now, then tighten the label when I have clean prior→future windows.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Search question:** Among pages with measurable search demand, which ones should a content editor review first for refresh / protect / expand / CTR or engagement work / monitoring?

**Unit of analysis:** one content page (`content_id`), scored inside a client (`client_id` used for grouping and holdout, never as a feature).

**Output:** a ranked action queue — priority score, suggested action, and reason codes a reviewer can inspect.

**Who acts:** a content editor or SEO reviewer with limited weekly capacity (for example, top 20–50 pages).

**Action they take:** open the highest-ranked pages, read the reason codes, and choose an editorial next step (refresh copy, expand thin pages, rewrite title/meta, improve engagement, merge/prune candidates, or deliberately monitor).

**Cost of a wrong call:**
- False positive (rank a healthy/noisy page high): wasted editor hours and opportunity cost — pages that actually need attention wait.
- False negative (bury a high-demand declining page): continued lost clicks/sessions while capacity is spent elsewhere.
- Overclaiming causation: treating “recommended for refresh” as “a refresh will recover traffic” risks bad process decisions; the safe use is review prioritization, not auto-publishing.

**Why data / ML (not just “train a model”):** a single if-statement rule (stale + visible, declining + demand, low CTR at a given position) already helps, but thousands of pages compete and the signals tangle — volume, age, freshness, position, CTR, engagement, content depth. A transparent baseline first, then a learned score only if it beats that baseline on a decision metric like Precision@K under client-holdout. The model earns its place if it surfaces better review candidates than the hand-written rules; if not, keep the rules.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
from pathlib import Path

import pandas as pd

# Repo-root relative path works from work/notebooks/ or the repo root.
candidates = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
]
data_path = next(p for p in candidates if p.exists())

df = pd.read_csv(data_path)

n_rows = len(df)
n_clients = df["client_id"].nunique()
decline_rate = (df["trend_direction"] == "down").mean()

# Starter-style "worth reviewing" slices (same spirit as docs baseline reason codes).
declining_with_demand = (
    (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
).sum()
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)  # ctr is ×100 percent: 0.5 means 0.5%, not 50%
).sum()
high_demand = df["impressions_90d"] >= 500
high_demand_decline_rate = (df.loc[high_demand, "trend_direction"] == "down").mean()

print(f"Starter rows × clients: {n_rows:,} pages across {n_clients} clients")
print(
    f"1) Declining proxy share (trend_direction == 'down'): "
    f"{decline_rate:.1%} of pages"
)
print(
    f"2) Declining-with-demand candidates (down & impressions_90d >= 100): "
    f"{declining_with_demand:,} pages — far more than a weekly review budget"
)
print(
    f"3) High-demand pages (impressions_90d >= 500): "
    f"{high_demand.sum():,} pages; declining share = {high_demand_decline_rate:.1%}"
)
print(
    f"   Bonus signal for queue mix — low-CTR visible pages "
    f"(imp>=500, position 1–20, ctr<0.5%): {low_ctr_visible:,}"
)
print()
print(
    "Why this lane is worth 7 weeks: ~13k declining-with-demand pages and ~10k "
    "low-CTR visible pages invent a ranking problem. A reviewer cannot scan them "
    "by hand; a baseline + honest ranked score can decide who gets scarce review time."
)
print(
    "Starter pipeline context (committed model_report.md): baseline Precision@50 "
    "= 0.24 vs random forest = 0.74 under client-holdout — learned ranking beat "
    "fixed rules on this proxy label. Capstone will need a stronger future-window "
    "label; the starter result only motivates the lane."
)

Starter rows × clients: 30,000 pages across 32 clients
1) Declining proxy share (trend_direction == 'down'): 54.2% of pages
2) Declining-with-demand candidates (down & impressions_90d >= 100): 13,152 pages — far more than a weekly review budget
3) High-demand pages (impressions_90d >= 500): 16,726 pages; declining share = 59.6%
   Bonus signal for queue mix — low-CTR visible pages (imp>=500, position 1–20, ctr<0.5%): 9,759

Why this lane is worth 7 weeks: ~13k declining-with-demand pages and ~10k low-CTR visible pages invent a ranking problem. A reviewer cannot scan them by hand; a baseline + honest ranked score can decide who gets scarce review time.
Starter pipeline context (committed model_report.md): baseline Precision@50 = 0.24 vs random forest = 0.74 under client-holdout — learned ranking beat fixed rules on this proxy label. Capstone will need a stronger future-window label; the starter result only motivates the lane.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim (if the later validation holds):**
- Observed associations between safe signals (impressions, position, age/freshness, CTR, engagement, depth) and a carefully defined outcome or review priority.
- Directional evidence that a ranked queue beats (or fails to beat) a transparent baseline on Precision@K / average precision under client-holdout or time-aware validation.
- Decision-support recommendations: “review these pages first, for these reason codes,” with confidence tied to volume and evidence.

**What I cannot claim:**
- That refreshing a page *causes* recovery — that needs an experiment or other causal design, not observational ranking alone.
- That I have found Google ranking factors, SERP algorithm levers, or AI citations/rankings.
- That the starter `is_declining_label` (from current-window `trend_direction`) is the true capstone target — it is a proxy that teaches the workflow; a stronger project uses prior-window features → future-window outcomes.
- That any single score replaces human editorial judgment — the queue is a reviewer aid.

**One-paragraph frame:** For content editors deciding which existing pages to review first under limited capacity, I will build a ranked action queue from observed search/content signals, scoring review priority against an observed decline/opportunity label measured by Precision@K (and average precision) under client-grouped validation. A wrong call wastes editor hours or buries a high-demand risk. A plain rule is not enough because thousands of candidates and tangled signals compete for the same shortlist. I will claim only observed, directional, decision-support results — not causal refresh payoffs or “predicting Google.”

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.